In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"qorxmazahbazl","key":"015cc426ef78862a59fe7291a6ec1f02"}'}

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#!/bin/bash
!kaggle datasets download sujaykapadnis/emotion-recognition-dataset

Dataset URL: https://www.kaggle.com/datasets/sujaykapadnis/emotion-recognition-dataset
License(s): CC-BY-NC-SA-4.0
emotion-recognition-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
import zipfile
with zipfile.ZipFile('/content/emotion-recognition-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
import shutil
shutil.rmtree('/content/dataset/Ahegao')

In [ ]:
import os
for dirpath, dirnames, filenames in os.walk('/content/dataset'):
    print(f'There are {len(dirnames)} directories ans {len(filenames)} images in {dirpath}.')

There are 5 directories ans 0 images in /content/dataset.
There are 0 directories ans 3740 images in /content/dataset/Happy.
There are 0 directories ans 1313 images in /content/dataset/Angry.
There are 0 directories ans 1234 images in /content/dataset/Surprise.
There are 0 directories ans 3934 images in /content/dataset/Sad.
There are 0 directories ans 4027 images in /content/dataset/Neutral.


In [ ]:
import tensorflow as tf
IMG_SIZE = (224, 224)
data_path = '/content/dataset'

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_path,
    validation_split = 0.2,
    subset = 'training',
    seed = 42,
    image_size = IMG_SIZE,
    crop_to_aspect_ratio = True
)
test_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_path,
    validation_split = 0.2,
    subset = 'validation',
    seed = 42,
    image_size = IMG_SIZE,
    crop_to_aspect_ratio = True
)

Found 14248 files belonging to 5 classes.
Using 11399 files for training.
Found 14248 files belonging to 5 classes.
Using 2849 files for validation.


In [ ]:
class_names = train_data.class_names
class_names

['Angry', 'Happy', 'Neutral', 'Sad', 'Surprise']

In [ ]:
train_data = train_data.cache('/tmp/train_cahce').shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
test_data = test_data.cache('/tmp/train_cahce').prefetch(buffer_size = tf.data.AUTOTUNE)

In [ ]:
n_classes = len(class_names)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape =(224, 224, 3)),
    tf.keras.layers.Rescaling(1/255),
    tf.keras.layers.Conv2D(filters = 64, kernel_size = 7, strides = 3, padding = 'same',
                           activation = 'relu', kernel_initializer = 'he_normal'),

    tf.keras.layers.MaxPool2D(pool_size = 2),
    tf.keras.layers.Conv2D(filters = 128, kernel_size = 3, padding = 'same',
                           activation = 'relu', kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters = 128, kernel_size = 3, padding = 'same',
                           activation = 'relu', kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size = 2),
    tf.keras.layers.Conv2D(filters = 256, kernel_size = 3, padding = 'same',
                           activation = 'relu', kernel_initializer='he_normal'),
    tf.keras.layers.Conv2D(filters = 256, kernel_size = 3, padding = 'same',
                           activation = 'relu', kernel_initializer='he_normal'),
    tf.keras.layers.MaxPool2D(pool_size = 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128, activation = 'relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=64, activation = 'relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(n_classes, activation = 'softmax')


])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 75, 75, 64)     │         9,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 37, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 37, 37, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 18, 18, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 18, 18, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20736)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,654,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,779,077 (14.42 MB)

 Trainable params: 3,779,077 (14.42 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    loss = 'sparse_categorical_crossentropy',
    optimizer = tf.keras.optimizers.Adam(),
    metrics = ['accuracy']
)

model.fit(train_data, epochs = 5, validation_data = test_data)

Epoch 1/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 60s 138ms/step - accuracy: 0.2792 - loss: 1.5428 - val_accuracy: 0.3001 - val_loss: 1.4775
Epoch 2/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 44s 122ms/step - accuracy: 0.2918 - loss: 1.4707 - val_accuracy: 0.3394 - val_loss: 1.4263
Epoch 3/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 82s 121ms/step - accuracy: 0.3369 - loss: 1.4164 - val_accuracy: 0.4616 - val_loss: 1.2619
Epoch 4/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 45s 125ms/step - accuracy: 0.4496 - loss: 1.2812 - val_accuracy: 0.5033 - val_loss: 1.1596
Epoch 5/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 83s 128ms/step - accuracy: 0.4910 - loss: 1.1896 - val_accuracy: 0.5433 - val_loss: 1.1146


In [ ]:
input_layer = tf.keras.layers.Input(shape = (224, 224, 3))

x = tf.keras.applications.resnet50.preprocess_input(input_layer)
base_model = tf.keras.applications.ResNet50(include_top = False, input_tensor = x)

avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(n_classes, activation = 'softmax')(avg)

model = tf.keras.Model(inputs = input_layer, outputs = output)

In [ ]:
base_model.trainable = False

In [ ]:
model.save("emotions_predictions.keras")


In [ ]:
from google.colab import files
files.download("emotions_predictions.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>